# Stage-1 SAM + GNN Refiner (COCO Minitrain)


In [ ]:
# Setup
import sys
import shutil
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ON_KAGGLE = Path("/kaggle/input").exists()
print("Kaggle:", ON_KAGGLE)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools", "opencv-python-headless"], check=False)

try:
    import segment_anything  # noqa: F401
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/segment-anything.git"],
        check=True,
    )


## Config

In [ ]:
CONFIG = {
    "data": {"img_size": 1024, "mask_size": 256, "num_workers": 2, "max_instances": 2000 if ON_KAGGLE else None},
    "model": {"sam_channels": 256, "feat_dim": 128, "hidden_dim": 64, "out_dim": 64, "grid_size": 32,
              "mask_size": 256, "num_gnn_layers": 2, "connectivity": "grid", "k_neighbors": 8},
    "training": {"batch_size": 2, "epochs": 5, "lr": 1e-4, "weight_decay": 1e-4, "bce_weight": 1.0, "dice_weight": 1.0},
}


## Dataset & Dataloader

In [ ]:
"""
COCO 2017 minitrain instance dataset for Stage-1 SAM+GNN training.

Paths match data/EDA.ipynb and Kaggle dataset layout.
"""

from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

try:
    from pycocotools import mask as mask_utils
except ImportError:
    mask_utils = None


def load_image_ids_from_txt(txt_path: Path) -> set:
    ids = set()
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            match = re.search(r"(\d{12})", line)
            if match:
                ids.add(int(match.group(1)))
    return ids


def filter_coco_json(data: dict, image_ids: set) -> dict:
    images = [img for img in data["images"] if img["id"] in image_ids]
    valid_ids = {img["id"] for img in images}
    annotations = [ann for ann in data["annotations"] if ann["image_id"] in valid_ids]
    return {
        "images": images,
        "annotations": annotations,
        "categories": data.get("categories", []),
    }


def ann_to_mask(ann: dict, height: int, width: int) -> np.ndarray:
    """Decode COCO annotation to binary HxW uint8 mask."""
    seg = ann["segmentation"]
    if isinstance(seg, list):
        if mask_utils is not None:
            rle = mask_utils.frPyObjects(seg, height, width)
            m = mask_utils.decode(mask_utils.merge(rle))
            if m.ndim == 3:
                m = m.max(axis=2)
            return m.astype(np.uint8)
        try:
            import cv2
        except ImportError as e:
            raise ImportError(
                "pycocotools or opencv-python required for polygon masks"
            ) from e
        mask = np.zeros((height, width), dtype=np.uint8)
        for poly in seg:
            pts = np.array(poly, dtype=np.float32).reshape(-1, 2)
            pts = np.round(pts).astype(np.int32)
            cv2.fillPoly(mask, [pts], 1)
        return mask
    if isinstance(seg, dict) and mask_utils:
        m = mask_utils.decode(seg)
        if m.ndim == 3:
            m = m.max(axis=2)
        return m.astype(np.uint8)
    raise ValueError("Unsupported segmentation format")


class CocoSamStage1Dataset(Dataset):
    """
    One sample = one COCO instance (image crop region implied by full image + mask).

    Returns:
        image: float tensor [3, img_size, img_size] in [0, 1]
        mask: float tensor [1, mask_size, mask_size] in {0, 1}
        meta: dict with image_id, ann_id (for debugging)
    """

    def __init__(
        self,
        coco_root: Path,
        ann_json: Path,
        image_id_txt: Path,
        split: str = "train",
        img_size: int = 1024,
        mask_size: int = 256,
        max_instances: Optional[int] = None,
    ):
        self.coco_root = Path(coco_root)
        self.img_size = img_size
        self.mask_size = mask_size
        self.split = split

        image_dir = self.coco_root / f"{split}2017"
        if not image_dir.exists():
            image_dir = self.coco_root / "images" / f"{split}2017"

        self.image_dir = image_dir

        with open(ann_json, "r", encoding="utf-8") as f:
            full_data = json.load(f)

        subset_ids = load_image_ids_from_txt(image_id_txt)
        data = filter_coco_json(full_data, subset_ids)

        self.images = {img["id"]: img for img in data["images"]}
        self.samples: List[Tuple[int, dict]] = []
        for ann in data["annotations"]:
            if ann.get("iscrowd", 0):
                continue
            img_id = ann["image_id"]
            if img_id not in self.images:
                continue
            self.samples.append((img_id, ann))

        if max_instances is not None and max_instances < len(self.samples):
            rng = np.random.RandomState(42)
            idx = rng.choice(len(self.samples), size=max_instances, replace=False)
            self.samples = [self.samples[i] for i in idx]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        img_id, ann = self.samples[idx]
        info = self.images[img_id]
        file_name = info["file_name"]
        path = self.image_dir / file_name
        if not path.exists():
            path = self.image_dir / Path(file_name).name

        image = Image.open(path).convert("RGB")
        w, h = image.size

        mask_np = ann_to_mask(ann, info["height"], info["width"])
        mask_img = Image.fromarray((mask_np * 255).astype(np.uint8))

        image = image.resize((self.img_size, self.img_size), Image.BILINEAR)
        mask_img = mask_img.resize((self.mask_size, self.mask_size), Image.NEAREST)

        image_t = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
        mask_t = torch.from_numpy(np.array(mask_img)).float().unsqueeze(0) / 255.0
        mask_t = (mask_t > 0.5).float()

        meta = {"image_id": img_id, "ann_id": ann["id"]}
        return image_t, mask_t, meta


def collate_stage1(batch):
    images = torch.stack([b[0] for b in batch], dim=0)
    masks = torch.stack([b[1] for b in batch], dim=0)
    meta = [b[2] for b in batch]
    return images, masks, meta


def build_coco_paths(
    kaggle: bool = True,
    coco_root: Optional[Path] = None,
) -> Dict[str, Path]:
    """Default paths for Kaggle vs local."""
    if kaggle:
        coco_root = Path(
            "/kaggle/input/datasets/awsaf49/coco-2017-dataset/coco2017"
        )
        ann_dir = coco_root / "annotations"
        minitrain_root = Path(
            "/kaggle/input/datasets/banuprasadb/coco-minitrain-10k/coco_minitrain_10k"
        )
        return {
            "coco_root": coco_root,
            "train_ann": ann_dir / "instances_train2017.json",
            "val_ann": ann_dir / "instances_val2017.json",
            "train_txt": minitrain_root / "train2017.txt",
            "val_txt": minitrain_root / "val2017.txt",
        }

    root = coco_root or Path("./data/coco2017")
    return {
        "coco_root": root,
        "train_ann": root / "annotations" / "instances_train2017.json",
        "val_ann": root / "annotations" / "instances_val2017.json",
        "train_txt": root / "minitrain" / "train2017.txt",
        "val_txt": root / "minitrain" / "val2017.txt",
    }


def get_stage1_dataloaders(
    config: dict,
    kaggle: bool = True,
) -> Tuple[DataLoader, DataLoader]:
    data_cfg = config.get("data", {})
    paths = build_coco_paths(kaggle=kaggle, coco_root=data_cfg.get("coco_root"))

    common = dict(
        coco_root=paths["coco_root"],
        img_size=data_cfg.get("img_size", 1024),
        mask_size=data_cfg.get("mask_size", 256),
        max_instances=data_cfg.get("max_instances"),
    )

    train_ds = CocoSamStage1Dataset(
        ann_json=paths["train_ann"],
        image_id_txt=paths["train_txt"],
        split="train",
        **common,
    )
    val_ds = CocoSamStage1Dataset(
        ann_json=paths["val_ann"],
        image_id_txt=paths["val_txt"],
        split="val",
        **common,
    )

    training_cfg = config.get("training", {})
    batch_size = training_cfg.get("batch_size", 2)
    num_workers = data_cfg.get("num_workers", 2)

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        collate_fn=collate_stage1,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False,
        collate_fn=collate_stage1,
    )
    return train_loader, val_loader


## GNN Refiner Model

In [ ]:
"""
Stage-1 GNN refiner over frozen SAM image embeddings (64x64x256).

Designed for Kaggle notebooks: pure PyTorch, no package imports required when
cells are copied inline. Matches PLAN.md tensor shapes (SAM stride-16, mask 256).
"""

from __future__ import annotations

from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class EdgeMLP(nn.Module):
    """MLP for edge weights from node pair features and relative position."""

    def __init__(self, in_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2 * in_dim + 2, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, 1),
        )

    def forward(
        self, fi: torch.Tensor, fj: torch.Tensor, delta_pos: torch.Tensor
    ) -> torch.Tensor:
        edge_input = torch.cat([fi, fj, delta_pos], dim=-1)
        return self.mlp(edge_input)


class NodeMLP(nn.Module):
    """MLP for node update after message aggregation."""

    def __init__(self, in_dim: int, hidden_dim: int, out_dim: int):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.mlp(x)


class GNNStage(nn.Module):
    """
    One round of attention-weighted message passing on a spatial graph.

    connectivity: 'grid' (8-neighbor) or 'local' (k-NN in normalized coordinates).
    """

    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        connectivity: str = "grid",
        k_neighbors: int = 8,
    ):
        super().__init__()
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.connectivity = connectivity
        self.k_neighbors = k_neighbors

        self.edge_mlp = EdgeMLP(in_dim, hidden_dim)
        self.node_mlp = NodeMLP(in_dim, hidden_dim, out_dim)
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

        self._edge_cache: dict[Tuple[int, int, str, int, str], Tuple[torch.Tensor, torch.Tensor]] = {}

    def _get_positions(self, h: int, w: int, device: torch.device) -> torch.Tensor:
        y = torch.linspace(0, 1, h, device=device)
        x = torch.linspace(0, 1, w, device=device)
        yy, xx = torch.meshgrid(y, x, indexing="ij")
        return torch.stack([xx.flatten(), yy.flatten()], dim=-1)

    def _build_edges_grid(
        self, h: int, w: int, device: torch.device
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        src_list, dst_list = [], []
        for i in range(h):
            for j in range(w):
                node_idx = i * w + j
                for di in (-1, 0, 1):
                    for dj in (-1, 0, 1):
                        if di == 0 and dj == 0:
                            continue
                        ni, nj = i + di, j + dj
                        if 0 <= ni < h and 0 <= nj < w:
                            src_list.append(node_idx)
                            dst_list.append(ni * w + nj)
        src = torch.tensor(src_list, dtype=torch.long, device=device)
        dst = torch.tensor(dst_list, dtype=torch.long, device=device)
        return src, dst

    def _build_edges_local(
        self, h: int, w: int, k: int, device: torch.device
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        num_nodes = h * w
        pos = self._get_positions(h, w, device)
        dist = torch.cdist(pos, pos)
        dist.fill_diagonal_(float("inf"))
        _, indices = dist.topk(k, dim=1, largest=False)
        src = torch.arange(num_nodes, device=device).repeat_interleave(k)
        dst = indices.flatten()
        return src, dst

    def _get_edges(
        self, h: int, w: int, device: torch.device
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        key = (h, w, self.connectivity, self.k_neighbors, str(device))
        if key not in self._edge_cache:
            if self.connectivity == "local":
                edges = self._build_edges_local(h, w, self.k_neighbors, device)
            else:
                edges = self._build_edges_grid(h, w, device)
            self._edge_cache[key] = edges
        return self._edge_cache[key]

    def forward(self, features: torch.Tensor, h: int, w: int) -> torch.Tensor:
        B, N, D = features.shape
        device = features.device
        pos = self._get_positions(h, w, device)
        src, dst = self._get_edges(h, w, device)
        delta_pos = pos[dst] - pos[src]

        out_features = []
        for b in range(B):
            f = features[b]
            fi, fj = f[src], f[dst]
            edge_weights = self.edge_mlp(fi, fj, delta_pos).squeeze(-1)
            edge_weights_exp = torch.exp(edge_weights)

            messages = torch.zeros(N, D, device=device)
            weights_sum = torch.zeros(N, device=device)
            weighted_fj = edge_weights_exp.unsqueeze(-1) * fj
            messages.index_add_(0, src, weighted_fj)
            weights_sum.index_add_(0, src, edge_weights_exp)
            messages = messages / (weights_sum.unsqueeze(-1) + 1e-8)

            updated = self.node_mlp(f + messages)
            updated = updated + self.proj(f)
            out_features.append(updated)

        return torch.stack(out_features, dim=0)


class SamStage1Refiner(nn.Module):
    """
    Lightweight GNN refiner on SAM embeddings.

    Args:
        sam_channels: SAM embedding depth (256 for ViT-B).
        grid_size: spatial grid for GNN (32 recommended for Kaggle memory).
        mask_size: output mask resolution (256 per PLAN Stage-1).
    """

    def __init__(
        self,
        sam_channels: int = 256,
        feat_dim: int = 128,
        hidden_dim: int = 64,
        out_dim: int = 64,
        grid_size: int = 32,
        mask_size: int = 256,
        num_gnn_layers: int = 2,
        connectivity: str = "grid",
        k_neighbors: int = 8,
    ):
        super().__init__()
        self.grid_size = grid_size
        self.mask_size = mask_size
        self.sam_spatial = 64

        self.input_proj = nn.Sequential(
            nn.Conv2d(sam_channels, feat_dim, kernel_size=1, bias=False),
            nn.BatchNorm2d(feat_dim),
            nn.ReLU(inplace=True),
        )

        stages: List[GNNStage] = []
        for i in range(num_gnn_layers):
            in_d = feat_dim if i == 0 else out_dim
            stages.append(
                GNNStage(
                    in_dim=in_d,
                    hidden_dim=hidden_dim,
                    out_dim=out_dim,
                    connectivity=connectivity,
                    k_neighbors=k_neighbors,
                )
            )
        self.stages = nn.ModuleList(stages)

        self.mask_head = nn.Sequential(
            nn.Conv2d(out_dim, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, sam_embed: torch.Tensor) -> torch.Tensor:
        """
        Args:
            sam_embed: [B, 256, 64, 64] from frozen SAM image encoder.

        Returns:
            mask_logits: [B, 1, mask_size, mask_size]
        """
        B = sam_embed.shape[0]
        x = sam_embed
        if x.shape[-1] != self.grid_size:
            x = F.adaptive_avg_pool2d(x, (self.grid_size, self.grid_size))

        x = self.input_proj(x)
        feats = x.flatten(2).transpose(1, 2)
        h = w = self.grid_size
        for stage in self.stages:
            feats = stage(feats, h, w)

        spatial = feats.transpose(1, 2).reshape(B, -1, h, w)
        logits = self.mask_head(spatial)
        logits = F.interpolate(
            logits,
            size=(self.mask_size, self.mask_size),
            mode="bilinear",
            align_corners=False,
        )
        return logits


def build_sam_stage1_refiner(config: Optional[dict] = None) -> SamStage1Refiner:
    """Build refiner from optional config dict."""
    cfg = config or {}
    model_cfg = cfg.get("model", cfg)
    return SamStage1Refiner(
        sam_channels=model_cfg.get("sam_channels", 256),
        feat_dim=model_cfg.get("feat_dim", 128),
        hidden_dim=model_cfg.get("hidden_dim", 64),
        out_dim=model_cfg.get("out_dim", 64),
        grid_size=model_cfg.get("grid_size", 32),
        mask_size=model_cfg.get("mask_size", 256),
        num_gnn_layers=model_cfg.get("num_gnn_layers", 2),
        connectivity=model_cfg.get("connectivity", "grid"),
        k_neighbors=model_cfg.get("k_neighbors", 8),
    )


def count_parameters(model: nn.Module, trainable_only: bool = True) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


## Losses, Metrics & SAM Helpers

In [ ]:
"""
Losses, metrics, and SAM preprocessing helpers for Stage-1 training.
"""

from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# #region agent log
def _agent_dbg_log(
    location: str,
    message: str,
    data: dict,
    hypothesis_id: str,
    run_id: str = "pre-fix",
) -> None:
    payload = {
        "sessionId": "5ff431",
        "timestamp": int(time.time() * 1000),
        "location": location,
        "message": message,
        "data": data,
        "hypothesisId": hypothesis_id,
        "runId": run_id,
    }
    line = json.dumps(payload) + "\n"
    candidates = [
        Path("debug-5ff431.log"),
        Path("/kaggle/working/debug-5ff431.log"),
    ]
    try:
        candidates.append(Path(__file__).resolve().parents[2] / "debug-5ff431.log")
    except NameError:
        pass
    for path in candidates:
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
            with open(path, "a", encoding="utf-8") as f:
                f.write(line)
            break
        except OSError:
            continue


def probe_cuda_runtime() -> bool:
    """
    Test the same CUDA ops used in preprocess_sam_batch.
    Returns False when PyTorch has no kernels for the GPU (common on new Kaggle GPUs).
    """
    if not torch.cuda.is_available():
        _agent_dbg_log(
            "sam_stage1_common.probe_cuda_runtime",
            "cuda not available",
            {"torch": torch.__version__},
            "A",
        )
        return False

    info = {
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
        "device_name": torch.cuda.get_device_name(0),
        "capability": list(torch.cuda.get_device_capability(0)),
    }
    try:
        x = torch.zeros(1, 3, 8, 8, device="cuda")
        x = F.interpolate(x, size=(16, 16), mode="bilinear", align_corners=False)
        mean = torch.tensor([123.675, 116.28, 103.53], device="cuda")
        std = torch.tensor([58.395, 57.12, 57.375], device="cuda")
        _ = (x - mean.view(1, 3, 1, 1)) / std.view(1, 3, 1, 1)
        torch.cuda.synchronize()
        _agent_dbg_log(
            "sam_stage1_common.probe_cuda_runtime",
            "cuda probe ok",
            info,
            "A",
        )
        return True
    except Exception as e:
        info["error"] = type(e).__name__
        info["error_msg"] = str(e)[:500]
        _agent_dbg_log(
            "sam_stage1_common.probe_cuda_runtime",
            "cuda probe failed",
            info,
            "A",
        )
        return False


def resolve_device(prefer_cuda: bool = True) -> torch.device:
    """Pick cuda only if runtime probe succeeds (Hypothesis A/D)."""
    use_cuda = prefer_cuda and probe_cuda_runtime()
    dev = torch.device("cuda" if use_cuda else "cpu")
    _agent_dbg_log(
        "sam_stage1_common.resolve_device",
        "resolved device",
        {"device": str(dev), "prefer_cuda": prefer_cuda},
        "D",
    )
    return dev
# #endregion


class DiceLoss(nn.Module):
    def __init__(self, smooth: float = 1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        pred = torch.sigmoid(pred)
        pred_flat = pred.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        union = pred_flat.sum() + target_flat.sum()
        dice = (2.0 * intersection + self.smooth) / (union + self.smooth)
        return 1.0 - dice


class CombinedSegLoss(nn.Module):
    """BCE + Dice (PLAN Stage-1 supervised loss)."""

    def __init__(self, bce_weight: float = 1.0, dice_weight: float = 1.0):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return self.bce_weight * self.bce(pred, target) + self.dice_weight * self.dice(
            pred, target
        )


def compute_iou(
    pred: torch.Tensor, target: torch.Tensor, threshold: float = 0.5
) -> float:
    if pred.min() < 0:
        pred = torch.sigmoid(pred)
    pred_binary = (pred > threshold).float()
    target_binary = (target > threshold).float()
    intersection = (pred_binary * target_binary).sum().item()
    union = pred_binary.sum().item() + target_binary.sum().item() - intersection
    if union == 0:
        return 1.0 if intersection == 0 else 0.0
    return intersection / union


def compute_dice(
    pred: torch.Tensor, target: torch.Tensor, threshold: float = 0.5
) -> float:
    if pred.min() < 0:
        pred = torch.sigmoid(pred)
    pred_binary = (pred > threshold).float()
    target_binary = (target > threshold).float()
    intersection = (pred_binary * target_binary).sum().item()
    total = pred_binary.sum().item() + target_binary.sum().item()
    if total == 0:
        return 1.0 if intersection == 0 else 0.0
    return 2.0 * intersection / total


class MetricTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.iou_sum = 0.0
        self.dice_sum = 0.0
        self.count = 0

    def update(self, pred: torch.Tensor, target: torch.Tensor):
        for i in range(pred.shape[0]):
            self.iou_sum += compute_iou(pred[i : i + 1], target[i : i + 1])
            self.dice_sum += compute_dice(pred[i : i + 1], target[i : i + 1])
            self.count += 1

    def compute(self) -> Dict[str, float]:
        if self.count == 0:
            return {"iou": 0.0, "dice": 0.0}
        return {
            "iou": self.iou_sum / self.count,
            "dice": self.dice_sum / self.count,
        }


def preprocess_sam_batch(
    images: torch.Tensor,
    pixel_mean: torch.Tensor,
    pixel_std: torch.Tensor,
    target_size: int = 1024,
) -> torch.Tensor:
    """
    Resize + pad to target_size and apply SAM normalization.

    Args:
        images: [B, 3, H, W] in [0, 1] RGB.

    Returns:
        [B, 3, target_size, target_size] normalized for SAM encoder.
    """
    B, C, H, W = images.shape
    scale = target_size / max(H, W)
    new_h, new_w = int(round(H * scale)), int(round(W * scale))
    x = F.interpolate(images, size=(new_h, new_w), mode="bilinear", align_corners=False)

    pad_h = target_size - new_h
    pad_w = target_size - new_w
    x = F.pad(x, (0, pad_w, 0, pad_h))
    mean = pixel_mean.view(1, 3, 1, 1).to(x.device, x.dtype)
    std = pixel_std.view(1, 3, 1, 1).to(x.device, x.dtype)
    return (x - mean) / std


@torch.no_grad()
def encode_sam_embeddings(
    sam_model: nn.Module,
    images: torch.Tensor,
    pixel_mean: torch.Tensor,
    pixel_std: torch.Tensor,
) -> torch.Tensor:
    """Run frozen SAM image encoder -> [B, 256, 64, 64]."""
    # #region agent log
    _agent_dbg_log(
        "sam_stage1_common.encode_sam_embeddings",
        "encode entry",
        {
            "images_device": str(images.device),
            "sam_param_device": str(next(sam_model.parameters()).device),
            "pixel_mean_device": str(pixel_mean.device),
        },
        "B",
    )
    # #endregion
    sam_model.eval()
    x = preprocess_sam_batch(images, pixel_mean, pixel_std, target_size=1024)
    out = sam_model.image_encoder(x)
    # #region agent log
    _agent_dbg_log(
        "sam_stage1_common.encode_sam_embeddings",
        "encode ok",
        {"out_shape": list(out.shape), "out_device": str(out.device)},
        "B",
    )
    # #endregion
    return out


def freeze_sam(sam_model: nn.Module) -> None:
    sam_model.eval()
    for p in sam_model.parameters():
        p.requires_grad = False


def load_sam_vit_b(checkpoint_path: str, device: torch.device):
    """Load SAM ViT-B (requires segment_anything installed)."""
    from segment_anything import sam_model_registry

    sam = sam_model_registry["vit_b"](checkpoint=checkpoint_path)
    sam.to(device)
    freeze_sam(sam)
    return sam


def get_sam_pixel_stats(device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    """SAM default ImageNet-style pixel mean/std buffers."""
    pixel_mean = torch.tensor([123.675, 116.28, 103.53], device=device)
    pixel_std = torch.tensor([58.395, 57.12, 57.375], device=device)
    return pixel_mean, pixel_std


## Device (CUDA runtime probe)

In [ ]:
DEVICE = resolve_device(prefer_cuda=True)
pixel_mean, pixel_std = get_sam_pixel_stats(DEVICE)
print("Training device:", DEVICE)
if str(DEVICE) == "cpu" and torch.cuda.is_available():
    print("WARNING: CUDA visible but probe failed; using CPU. Upgrade PyTorch for this GPU.")


## Load SAM ViT-B (frozen)

In [ ]:
from segment_anything import sam_model_registry

CHECKPOINT_NAME = "sam_vit_b_01ec64.pth"
CKPT_DIR = Path("/kaggle/working/checkpoints") if ON_KAGGLE else Path("./checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / CHECKPOINT_NAME

if not CKPT_PATH.exists() and ON_KAGGLE:
    matches = list(Path("/kaggle/input").rglob(CHECKPOINT_NAME))
    if matches:
        shutil.copyfile(matches[0], CKPT_PATH)

if not CKPT_PATH.exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", CKPT_PATH
    )

sam = sam_model_registry["vit_b"](checkpoint=str(CKPT_PATH))
sam.to(DEVICE)
freeze_sam(sam)
print("SAM loaded on", DEVICE)


## Build Model, DataLoaders, Optimizer

In [ ]:
train_loader, val_loader = get_stage1_dataloaders(CONFIG, kaggle=ON_KAGGLE)
print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)}")
refiner = build_sam_stage1_refiner(CONFIG).to(DEVICE)
print(f"GNN params: {count_parameters(refiner):,}")
criterion = CombinedSegLoss(CONFIG["training"]["bce_weight"], CONFIG["training"]["dice_weight"])
optimizer = torch.optim.AdamW(refiner.parameters(), lr=CONFIG["training"]["lr"], weight_decay=CONFIG["training"]["weight_decay"])


## Training & Validation

In [ ]:
def train_one_epoch(sam_model, refiner, loader, optimizer, criterion):
    refiner.train()
    total_loss, n_batches = 0.0, 0
    for images, masks, _ in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        with torch.no_grad():
            sam_embed = encode_sam_embeddings(sam_model, images, pixel_mean, pixel_std)
        logits = refiner(sam_embed)
        loss = criterion(logits, masks)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)

@torch.no_grad()
def validate(sam_model, refiner, loader, criterion):
    refiner.eval()
    total_loss, n_batches = 0.0, 0
    tracker = MetricTracker()
    for images, masks, _ in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        sam_embed = encode_sam_embeddings(sam_model, images, pixel_mean, pixel_std)
        logits = refiner(sam_embed)
        total_loss += criterion(logits, masks).item()
        n_batches += 1
        tracker.update(logits, masks)
    m = tracker.compute()
    m["loss"] = total_loss / max(n_batches, 1)
    return m

history = []
for epoch in range(1, CONFIG["training"]["epochs"] + 1):
    train_loss = train_one_epoch(sam, refiner, train_loader, optimizer, criterion)
    val_metrics = validate(sam, refiner, val_loader, criterion)
    history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})
    print(f"Epoch {epoch} | train={train_loss:.4f} val={val_metrics['loss']:.4f} IoU={val_metrics['iou']:.4f} Dice={val_metrics['dice']:.4f}")

save_dir = Path("/kaggle/working") if ON_KAGGLE else Path("./outputs")
save_dir.mkdir(parents=True, exist_ok=True)
torch.save({"config": CONFIG, "state_dict": refiner.state_dict(), "history": history}, save_dir / "sam_stage1_refiner.pt")


## Visualization

In [ ]:
@torch.no_grad()
def show_batch(n=3):
    refiner.eval()
    images, masks, _ = next(iter(val_loader))
    images, masks = images[:n].to(DEVICE), masks[:n].to(DEVICE)
    preds = (torch.sigmoid(refiner(encode_sam_embeddings(sam, images, pixel_mean, pixel_std))) > 0.5).float()
    fig, axes = plt.subplots(n, 3, figsize=(9, 3*n))
    if n == 1: axes = axes.reshape(1, -1)
    for i in range(n):
        axes[i,0].imshow(images[i].cpu().permute(1,2,0)); axes[i,0].set_title("Input")
        axes[i,1].imshow(masks[i,0].cpu(), cmap="gray"); axes[i,1].set_title("GT")
        axes[i,2].imshow(preds[i,0].cpu(), cmap="gray"); axes[i,2].set_title("Pred")
        for j in range(3): axes[i,j].axis("off")
    plt.tight_layout(); plt.show()
show_batch(3)
